# Wi-Fi Fingerprint Indoor Localization — Regression Track

**Objective:** Predict the indoor (X, Y) coordinates of a device using Wi-Fi RSSI fingerprints.

**Dataset:** UJIndoorLoc (UCI ML Repository) — 520 WAP signal readings from 3 buildings across multiple floors.

**Team:** K Ganesh Giridhar (519) · G R Balaji (510) · A Suhas Reddy (503)

---
## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("Libraries loaded successfully")

Libraries loaded successfully


## 2. Dataset Loading & Audit

Load the raw UJIndoorLoc training and validation sets. We keep the originals untouched in `data/raw/`.

In [2]:
train_df = pd.read_csv('../data/raw/trainingData.csv')
val_df   = pd.read_csv('../data/raw/validationData.csv')

print(f"Training set  : {train_df.shape[0]} rows, {train_df.shape[1]} columns")
print(f"Validation set: {val_df.shape[0]} rows, {val_df.shape[1]} columns")

Training set  : 19937 rows, 529 columns
Validation set: 1111 rows, 529 columns


**Observation:** Training set has ~19,937 samples and 529 columns (520 WAPs + 9 metadata). Validation has 1,111 samples.

In [3]:
# Column types overview
train_df.info(verbose=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19937 entries, 0 to 19936
Columns: 529 entries, WAP001 to TIMESTAMP
dtypes: float64(2), int64(527)
memory usage: 80.5 MB


In [4]:
# First few rows — WAP columns and target columns
print("WAP columns sample:")
print(train_df.iloc[:3, :5])
print()
print("Target / metadata columns:")
print(train_df[['LONGITUDE','LATITUDE','FLOOR','BUILDINGID','SPACEID']].head())

WAP columns sample:
   WAP001  WAP002  WAP003  WAP004  WAP005
0     100     100     100     100     100
1     100     100     100     100     100
2     100     100     100     100     100

Target / metadata columns:
   LONGITUDE      LATITUDE  FLOOR  BUILDINGID  SPACEID
0 -7541.2643  4.864921e+06      2           1      106
1 -7536.6212  4.864934e+06      2           1      106
2 -7519.1524  4.864950e+06      2           1      103
3 -7524.5704  4.864934e+06      2           1      102
4 -7632.1436  4.864982e+06      0           0      122


In [5]:
# Check for missing values
missing = train_df.isnull().sum().sum()
print(f"Total missing values in training set: {missing}")

Total missing values in training set: 0


**Inference:** No traditional missing values (NaN) exist. However, the RSSI value `+100` is a **sentinel** meaning 'AP not detected' — this needs special treatment later.

In [6]:
# WAP columns
wap_cols = [c for c in train_df.columns if c.startswith('WAP')]
print(f"Number of WAP features: {len(wap_cols)}")

# Check how many +100 values exist
sentinel_count = (train_df[wap_cols] == 100).sum().sum()
total_cells = train_df[wap_cols].shape[0] * train_df[wap_cols].shape[1]
pct = (sentinel_count / total_cells) * 100
print(f"Sentinel (+100) values: {sentinel_count:,} out of {total_cells:,} ({pct:.1f}%)")

Number of WAP features: 520


Sentinel (+100) values: 10,008,477 out of 10,367,240 (96.5%)

**Key Finding:** A very large proportion of WAP readings are +100 (not detected). This is expected — a device can only see a small subset of all 520 access points from any location.

In [7]:
# Target variable distributions
print("=== LONGITUDE ===")
print(train_df['LONGITUDE'].describe())
print()
print("=== LATITUDE ===")
print(train_df['LATITUDE'].describe())
print()
print("=== FLOOR distribution ===")
print(train_df['FLOOR'].value_counts().sort_index())
print()
print("=== BUILDING distribution ===")
print(train_df['BUILDINGID'].value_counts().sort_index())

=== LONGITUDE ===
count    19937.000000
mean     -7464.275947
std        123.402010
min      -7691.338400
25%      -7594.737000
50%      -7423.060900
75%      -7359.193000
max      -7300.818990
Name: LONGITUDE, dtype: float64

=== LATITUDE ===
count    1.993700e+04
mean     4.864871e+06
std      6.693318e+01
min      4.864746e+06
25%      4.864821e+06
50%      4.864852e+06
75%      4.864930e+06
max      4.865017e+06
Name: LATITUDE, dtype: float64

=== FLOOR distribution ===
FLOOR
0    4369
1    5002
2    4416
3    5048
4    1102
Name: count, dtype: int64

=== BUILDING distribution ===
BUILDINGID
0    5249
1    5196
2    9492
Name: count, dtype: int64


**Observation:**
- Coordinates span a wide range (campus-level) across 3 buildings
- Floor values range from 0 to 4 (5 floors total)
- Building 0, 1, 2 have varying sample counts — slightly imbalanced but acceptable

---
## 3. Exploratory Data Analysis

### 3.1 Target Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Longitude
axes[0, 0].hist(train_df['LONGITUDE'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Longitude Distribution')
axes[0, 0].set_xlabel('Longitude')
axes[0, 0].set_ylabel('Count')

# Latitude
axes[0, 1].hist(train_df['LATITUDE'], bins=50, color='coral', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Latitude Distribution')
axes[0, 1].set_xlabel('Latitude')
axes[0, 1].set_ylabel('Count')

# Floor
train_df['FLOOR'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 0], color='seagreen', edgecolor='black')
axes[1, 0].set_title('Floor Distribution')
axes[1, 0].set_xlabel('Floor')
axes[1, 0].set_ylabel('Count')

# Building
train_df['BUILDINGID'].value_counts().sort_index().plot(kind='bar', ax=axes[1, 1], color='darkorange', edgecolor='black')
axes[1, 1].set_title('Building Distribution')
axes[1, 1].set_xlabel('Building ID')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

**Inference:** Longitude and Latitude show multi-modal distributions — samples are clustered around specific building regions. Floor 0, 1, 2 have more samples while floors 3 and 4 are less represented.

### 3.2 Spatial Distribution (Scatter Plot)

In [ ]:
plt.figure(figsize=(10, 8))
scatter = plt.scatter(train_df['LONGITUDE'], train_df['LATITUDE'],
                      c=train_df['BUILDINGID'], cmap='Set1', alpha=0.4, s=10)
plt.colorbar(scatter, label='Building ID')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Spatial Distribution of Samples by Building')
plt.tight_layout()
plt.show()

**Inference:** The three buildings are clearly separated in coordinate space. This is good — the model can learn distinct spatial patterns for each building.

### 3.3 Feature-Target Scatter Plots

In [10]:
# Find the top 5 most frequently detected WAPs (least +100 values)
detected_counts = (train_df[wap_cols] != 100).sum().sort_values(ascending=False)
top_waps = detected_counts.head(5).index.tolist()
print("Top 5 most commonly detected WAPs:", top_waps)

Top 5 most commonly detected WAPs: ['WAP496', 'WAP087', 'WAP502', 'WAP517', 'WAP078']


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# WAP vs Longitude
wap_example = top_waps[0]
mask = train_df[wap_example] != 100
axes[0].scatter(train_df.loc[mask, wap_example], train_df.loc[mask, 'LONGITUDE'],
                alpha=0.3, s=8, color='steelblue')
axes[0].set_xlabel(f'{wap_example} RSSI')
axes[0].set_ylabel('Longitude')
axes[0].set_title(f'{wap_example} vs Longitude')

# WAP vs Latitude
axes[1].scatter(train_df.loc[mask, wap_example], train_df.loc[mask, 'LATITUDE'],
                alpha=0.3, s=8, color='coral')
axes[1].set_xlabel(f'{wap_example} RSSI')
axes[1].set_ylabel('Latitude')
axes[1].set_title(f'{wap_example} vs Latitude')

plt.tight_layout()
plt.show()

**Inference:** Individual WAPs show localized detection patterns — strong signal from a particular AP correlates with proximity to it. This confirms RSSI values carry useful spatial information.

### 3.4 Correlation Heatmap (Top Detected WAPs)

In [ ]:
# Correlation among top 15 WAPs and targets
top15 = detected_counts.head(15).index.tolist()
corr_cols = top15 + ['LONGITUDE', 'LATITUDE']

# Replace 100 with NaN for correlation calculation
corr_df = train_df[corr_cols].replace(100, np.nan)
corr_matrix = corr_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Correlation Heatmap — Top 15 WAPs + Targets')
plt.tight_layout()
plt.show()

**Inference:**
- Some WAPs show moderate correlation with LONGITUDE or LATITUDE — these will be important features
- Many WAPs are weakly correlated with each other — low multicollinearity is good
- The heatmap confirms that WAP signals encode spatial information differently per AP

### 3.5 Floor-wise Coordinate Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, bid in enumerate(sorted(train_df['BUILDINGID'].unique())):
    bdata = train_df[train_df['BUILDINGID'] == bid]
    scatter = axes[i].scatter(bdata['LONGITUDE'], bdata['LATITUDE'],
                              c=bdata['FLOOR'], cmap='viridis', alpha=0.5, s=8)
    axes[i].set_title(f'Building {bid}')
    axes[i].set_xlabel('Longitude')
    axes[i].set_ylabel('Latitude')
    plt.colorbar(scatter, ax=axes[i], label='Floor')

plt.suptitle('Coordinate Distribution by Building and Floor', y=1.02)
plt.tight_layout()
plt.show()

**Inference:** Within each building, different floors overlap in X-Y space but are vertically separated. This means coordinate regression alone won't identify the floor — we need a separate classifier for that (Classification track).

---
## 4. Data Cleaning

### 4.1 Sentinel Value Treatment

RSSI = +100 means the AP was not detected. We replace it with **-105 dBm** (below the typical detection threshold of -104 dBm). This preserves the ordinal relationship: weaker signals have more negative values.

In [14]:
# Replace sentinel +100 with -105
train_clean = train_df.copy()
val_clean = val_df.copy()

train_clean[wap_cols] = train_clean[wap_cols].replace(100, -105)
val_clean[wap_cols] = val_clean[wap_cols].replace(100, -105)

print("Sentinel replacement done")
print(f"RSSI range now: [{train_clean[wap_cols].min().min()}, {train_clean[wap_cols].max().max()}]")

Sentinel replacement done


RSSI range now: [-105, 0]


**Why -105?** The weakest detectable signal is around -104 dBm. Setting undetected APs to -105 keeps them below the detection floor without introducing a massive gap (like -999 would).

### 4.2 Remove Zero-Variance WAPs

Some WAPs may never be detected in the training set — they have constant values. These add no information.

In [15]:
# Find WAPs with zero variance (all same value after cleaning)
from sklearn.feature_selection import VarianceThreshold

variances = train_clean[wap_cols].var()
zero_var = variances[variances == 0].index.tolist()
print(f"WAPs with zero variance: {len(zero_var)}")

# Remove them
wap_cols_clean = [c for c in wap_cols if c not in zero_var]
print(f"WAPs remaining after removal: {len(wap_cols_clean)}")

WAPs with zero variance: 55
WAPs remaining after removal: 465


**Decision:** Removing zero-variance WAPs reduces dimensionality without losing any information. These APs were never detected by any sample.

### 4.3 Duplicate Check

In [16]:
dupes = train_clean.duplicated().sum()
print(f"Duplicate rows: {dupes}")

if dupes > 0:
    train_clean = train_clean.drop_duplicates()
    print(f"After removal: {train_clean.shape[0]} rows")
else:
    print("No duplicates found — good")

Duplicate rows: 637


After removal: 19300 rows


### 4.4 Outlier Check

In [17]:
# Check target variable outliers using IQR
for col in ['LONGITUDE', 'LATITUDE']:
    Q1 = train_clean[col].quantile(0.25)
    Q3 = train_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((train_clean[col] < lower) | (train_clean[col] > upper)).sum()
    print(f"{col}: {outliers} outliers (IQR method)")

LONGITUDE: 0 outliers (IQR method)
LATITUDE: 0 outliers (IQR method)


**Decision:** We keep the coordinate outliers since they represent real physical locations at the edges of the buildings. Removing them would lose valid spatial data points.

---
## 5. Feature Engineering

We create summary features from the raw RSSI fingerprint. These capture the overall signal environment at each location.

In [21]:
# Engineered features based on RSSI values
# Using -105 as 'not detected' threshold

def engineer_features(df, wap_columns):
    """Create RSSI summary features from WAP readings."""
    wap_data = df[wap_columns]

    # Count of detected APs (RSSI > -105)
    df['n_visible_aps'] = (wap_data > -105).sum(axis=1)

    # Mean RSSI of detected APs only
    detected = wap_data.where(wap_data > -105)
    df['mean_rssi'] = detected.mean(axis=1).fillna(-105)

    # Max (strongest) signal
    df['max_rssi'] = detected.max(axis=1).fillna(-105)

    # Std of detected signals
    df['std_rssi'] = detected.std(axis=1).fillna(0)

    # Range of detected signals
    df['rssi_range'] = (detected.max(axis=1) - detected.min(axis=1)).fillna(0)

    return df

train_clean = engineer_features(train_clean, wap_cols_clean)
val_clean = engineer_features(val_clean, wap_cols_clean)

print("Engineered features created:")
print(train_clean[['n_visible_aps','mean_rssi','max_rssi','std_rssi','rssi_range']].describe().round(2))
print()
print("NaN check:", train_clean[['n_visible_aps','mean_rssi','max_rssi','std_rssi','rssi_range']].isnull().sum().sum(), "NaN values")

Engineered features created:
       n_visible_aps  mean_rssi  max_rssi  std_rssi  rssi_range
count       19300.00   19300.00  19300.00  19300.00    19300.00
mean           17.88     -78.88    -57.11     10.95       35.08
std             7.32       6.08     13.41      3.74       13.01
min             0.00    -105.00   -105.00      0.00        0.00
25%            13.00     -82.50    -64.00      8.59       28.00
50%            17.00     -79.00    -58.00     10.95       35.00
75%            21.00     -75.57    -50.00     13.25       42.00
max            51.00     -51.05      0.00     35.92       95.00

NaN check: 0 NaN values


**Why these features?**
- `n_visible_aps`: How many APs a device can see varies by location — open areas see more APs
- `mean_rssi`: Overall signal strength indicates proximity to AP clusters
- `max_rssi`: The strongest signal usually comes from the nearest AP
- `std_rssi` and `rssi_range`: Signal variability captures whether the device is near many APs or just a few

These give the model a high-level summary alongside the detailed per-AP readings.

In [22]:
# Check correlation of new features with targets
eng_features = ['n_visible_aps', 'mean_rssi', 'max_rssi', 'std_rssi', 'rssi_range']

print("Correlation with LONGITUDE:")
for f in eng_features:
    corr = train_clean[f].corr(train_clean['LONGITUDE'])
    print(f"  {f}: {corr:.3f}")
print()
print("Correlation with LATITUDE:")
for f in eng_features:
    corr = train_clean[f].corr(train_clean['LATITUDE'])
    print(f"  {f}: {corr:.3f}")

Correlation with LONGITUDE:
  n_visible_aps: 0.231
  mean_rssi: -0.227
  max_rssi: -0.017
  std_rssi: -0.067
  rssi_range: 0.025

Correlation with LATITUDE:
  n_visible_aps: -0.339
  mean_rssi: 0.216
  max_rssi: -0.047
  std_rssi: 0.024
  rssi_range: -0.100


**Inference:** The engineered features show some correlation with coordinates. Even modest correlations can help — they provide the model with aggregate spatial cues.

---
## 6. Prepare Train / Validation Sets & Scaling

**Important:** We use the full training set for training and the separate **validation set** (`validationData.csv`) for evaluation. This avoids the inflated R² scores that result from splitting a single dataset — the validation set was collected at different times/conditions, giving a realistic estimate of generalisation performance. We fit the scaler on the training set only to prevent data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Feature columns = WAP columns + engineered features
feature_cols = wap_cols_clean + eng_features

# Use the FULL training set for training
X_train = train_clean[feature_cols]
y_lon_train = train_clean['LONGITUDE']
y_lat_train = train_clean['LATITUDE']

# Use the SEPARATE validation set for evaluation
X_test = val_clean[feature_cols]
y_lon_test = val_clean['LONGITUDE']
y_lat_test = val_clean['LATITUDE']

print(f"Training:   {X_train.shape[0]} samples")
print(f"Validation: {X_test.shape[0]} samples (from validationData.csv)")
print(f"Features:   {X_train.shape[1]}")

In [ ]:
# Scale features — fit on train only, transform both train and validation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Safety: fill any remaining NaN from scaling edge cases
X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0)
X_test_scaled  = np.nan_to_num(X_test_scaled, nan=0.0)

print("Scaling done (fitted on training set only)")
print(f"Train mean ≈ {X_train_scaled.mean():.6f} (should be ~0)")
print(f"Train std  ≈ {X_train_scaled.std():.4f}  (should be ~1)")
print(f"NaN in train: {np.isnan(X_train_scaled).sum()}")

**Why StandardScaler?** Algorithms like SVR, KNN, and regularized linear models (Ridge, Lasso) are sensitive to feature scale. StandardScaler centers features at mean=0 and std=1 without distorting the RSSI value distribution.

**Why a separate validation set?** The training and validation data were collected at different times and possibly under different conditions. Using this genuinely independent test set gives a much more realistic estimate of model performance than a random train/test split from the same collection.

---
## 7. Regression Models

We train all 10 required regression algorithms on the same train/test split. For simplicity we predict **LONGITUDE** as the primary target and report all metrics. The same pipeline applies to LATITUDE.

> **Evaluation metrics:** R², RMSE, MAE

In [25]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def evaluate_model(model, X_tr, X_te, y_tr, y_te, model_name):
    """Train, predict, and return metrics."""
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    r2   = r2_score(y_te, y_pred)
    rmse = np.sqrt(mean_squared_error(y_te, y_pred))
    mae  = mean_absolute_error(y_te, y_pred)

    print(f"{model_name:35s} | R²: {r2:.4f} | RMSE: {rmse:.2f} | MAE: {mae:.2f}")
    return {'Model': model_name, 'R2': round(r2, 4), 'RMSE': round(rmse, 2), 'MAE': round(mae, 2), 'predictions': y_pred}

results = []

### 7.1 Linear Regression

Baseline model — fits a straight hyperplane through the feature space.

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
res = evaluate_model(lr, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'Linear Regression')
results.append(res)

**Inference:** Linear Regression gives us a baseline R². If it's already high, the relationship between WAP signals and coordinates is fairly linear. If low, we need non-linear models.

### 7.2 Ridge Regression

L2 regularisation — penalises large coefficients to reduce overfitting, especially useful with 500+ features.

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0, random_state=42)
res = evaluate_model(ridge, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'Ridge Regression')
results.append(res)

**Inference:** Ridge typically performs similar to or slightly better than plain Linear Regression when features are many and possibly correlated. The L2 penalty keeps coefficients small.

### 7.3 Lasso Regression

L1 regularisation — can drive some coefficients to exactly zero, effectively doing feature selection.

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=1.0, random_state=42, max_iter=500, tol=0.01)
res = evaluate_model(lasso, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'Lasso Regression')
results.append(res)

# How many features got zeroed out?
n_zero = (lasso.coef_ == 0).sum()
print(f"Features with zero coefficient: {n_zero} out of {len(lasso.coef_)}")

**Inference:** Lasso's feature sparsity tells us how many WAPs are truly relevant for predicting longitude. If many coefficients are zero, most APs don't contribute to positioning from this direction.

### 7.4 ElasticNet Regression

Combines L1 and L2 penalties — a middle ground between Ridge and Lasso.

In [ ]:
from sklearn.linear_model import ElasticNet

enet = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42, max_iter=500, tol=0.01)
res = evaluate_model(enet, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'ElasticNet Regression')
results.append(res)

**Inference:** ElasticNet balances feature selection (L1) with coefficient shrinkage (L2). Useful when features are grouped — correlated WAPs from the same area get shared weight instead of one being zeroed.

### 7.5 Polynomial Regression

Apply polynomial feature transformation then fit linear regression. We use degree=2 as degree=3 with 500+ features would be computationally expensive.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# Use only top 20 features to keep polynomial expansion manageable
from sklearn.feature_selection import SelectKBest, f_regression

poly_pipe = Pipeline([
    ('select', SelectKBest(f_regression, k=20)),
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('reg', LinearRegression())
])

res = evaluate_model(poly_pipe, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'Polynomial Regression (deg=2)')
results.append(res)

**Inference:** Polynomial features capture non-linear relationships. We select top-20 features first to avoid the combinatorial explosion of polynomial terms with 500+ features. This is a practical trade-off between expressiveness and tractability.

### 7.6 Decision Tree Regressor

Non-linear model that partitions feature space into regions. Easy to interpret.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=15, random_state=42)
res = evaluate_model(dt, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'Decision Tree Regressor')
results.append(res)

**Inference:** Decision trees can capture complex non-linear patterns in RSSI data. However, they tend to overfit if max_depth is too large. We'll tune this later.

### 7.7 Random Forest Regressor

Ensemble of decision trees — reduces overfitting through bagging (bootstrap aggregation).

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
res = evaluate_model(rf, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'Random Forest Regressor')
results.append(res)

**Inference:** Random Forest usually outperforms a single decision tree by averaging out individual tree errors. It's also robust to noise — important for RSSI data which is inherently noisy.

### 7.8 Gradient Boosting Regressor

Sequential ensemble — each tree corrects errors of the previous one.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)
res = evaluate_model(gbr, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'Gradient Boosting Regressor')
results.append(res)


**Inference:** Gradient Boosting often achieves the best performance among traditional ML models. The sequential correction mechanism helps it learn fine-grained spatial patterns.

### 7.9 Support Vector Regressor (SVR)

SVR finds a hyperplane that fits most points within an epsilon-tube. Requires scaled features.

In [ ]:
from sklearn.svm import SVR

# SVR is slow on large datasets — use a subset if needed
if X_train_scaled.shape[0] > 10000:
    # Random sample for faster training
    np.random.seed(42)
    idx = np.random.choice(X_train_scaled.shape[0], 10000, replace=False)
    X_tr_svr = X_train_scaled[idx]
    y_tr_svr = y_lon_train.values[idx]
    print("Using 10,000 sample subset for SVR (full dataset too large)")
else:
    X_tr_svr = X_train_scaled
    y_tr_svr = y_lon_train.values

svr = SVR(kernel='rbf', C=100, epsilon=0.1)
res = evaluate_model(svr, X_tr_svr, X_test_scaled, y_tr_svr, y_lon_test, 'SVR (RBF kernel)')
results.append(res)

**Inference:** SVR with RBF kernel can model non-linear relationships but is computationally expensive on large datasets. We used a subset for tractability. Performance might improve with full data but training time grows quadratically.

### 7.10 K-Nearest Neighbors Regressor

Predicts by averaging the target values of the k closest training samples.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn = KNeighborsRegressor(n_neighbors=5, weights='distance', n_jobs=-1)
res = evaluate_model(knn, X_train_scaled, X_test_scaled, y_lon_train, y_lon_test, 'KNN Regressor (k=5)')
results.append(res)

**Inference:** KNN is naturally suited for fingerprint-based localization — similar Wi-Fi fingerprints should correspond to nearby locations. The 'distance' weighting gives closer neighbors more influence, which matches physical proximity.

---
## 8. Hyperparameter Tuning

We apply GridSearchCV on the two best-performing models from the initial comparison.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

# Sort results by R2 to find top 2
results_df = pd.DataFrame([{k:v for k,v in r.items() if k != 'predictions'} for r in results])
results_df_sorted = results_df.sort_values('R2', ascending=False)
print("Top models before tuning:")
print(results_df_sorted.head())

### 8.1 Tuning Random Forest

In [ ]:
rf_params = {
    'n_estimators': [50, 100],
    'max_depth': [15, 25],
    'min_samples_split': [2]
}

rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    rf_params,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
rf_grid.fit(X_train_scaled, y_lon_train)

print(f"Best RF params: {rf_grid.best_params_}")
print(f"Best RF CV R²:  {rf_grid.best_score_:.4f}")

rf_best_pred = rf_grid.predict(X_test_scaled)
rf_best_r2 = r2_score(y_lon_test, rf_best_pred)
rf_best_rmse = np.sqrt(mean_squared_error(y_lon_test, rf_best_pred))
print(f"RF test R²: {rf_best_r2:.4f} | RMSE: {rf_best_rmse:.2f}")

### 8.2 Tuning K-Nearest Neighbors (KNN)

In [ ]:
# Tune KNN because it was the best untuned model
knn_params = {
    'n_neighbors': [3, 5, 7],
    'weights': ['distance'],
    'p': [1, 2]
}

knn_grid = GridSearchCV(
    KNeighborsRegressor(n_jobs=-1),
    knn_params,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=0
)
knn_grid.fit(X_train_scaled, y_lon_train)

print(f"Best KNN params: {knn_grid.best_params_}")
print(f"Best KNN CV R²:  {knn_grid.best_score_:.4f}")

knn_best_pred = knn_grid.predict(X_test_scaled)
knn_best_r2 = r2_score(y_lon_test, knn_best_pred)
knn_best_rmse = np.sqrt(mean_squared_error(y_lon_test, knn_best_pred))
print(f"KNN test R²: {knn_best_r2:.4f} | RMSE: {knn_best_rmse:.2f}")

### 8.3 Tuning Gradient Boosting

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

# Three-minute fallback: one lightweight CV trial on a representative subset.
tune_size = min(2000, len(X_train_scaled))
tune_rng = np.random.RandomState(42)
tune_idx = tune_rng.choice(len(X_train_scaled), size=tune_size, replace=False)

X_tune = X_train_scaled[tune_idx]
y_tune = y_lon_train.iloc[tune_idx]

gb_params = {
    'max_iter': [50],
    'learning_rate': [0.1],
    'max_leaf_nodes': [15],
    'l2_regularization': [0.1]
}

gb_grid = RandomizedSearchCV(
    HistGradientBoostingRegressor(random_state=42, early_stopping=True),
    gb_params,
    n_iter=1,
    cv=2,
    scoring='r2',
    n_jobs=1,
    random_state=42,
    verbose=1
)
gb_grid.fit(X_tune, y_tune)

print(f"Best GBR params: {gb_grid.best_params_}")
print(f"Best GBR CV R²:  {gb_grid.best_score_:.4f}")

gb_best_pred = gb_grid.predict(X_test_scaled)
gb_best_r2 = r2_score(y_lon_test, gb_best_pred)
gb_best_rmse = np.sqrt(mean_squared_error(y_lon_test, gb_best_pred))
print(f"Test R²: {gb_best_r2:.4f} | Test RMSE: {gb_best_rmse:.2f}")

**Conclusion:** Hyperparameter tuning typically gives a modest improvement over default parameters. The best configuration balances model complexity (depth, estimators) with generalization.

### 8.4 Cross-Validation R² for Tuned Models

In [ ]:
from sklearn.model_selection import cross_val_score

# 5-fold CV R² for the tuned KNN and tuned tree-based models
print("5-Fold Cross-Validation R² scores:")
print()

cv_knn = cross_val_score(knn_grid.best_estimator_, X_train_scaled, y_lon_train, cv=5, scoring='r2', n_jobs=-1)
print(f"Tuned KNN:            {cv_knn.mean():.4f} ± {cv_knn.std():.4f}")
print(f"  Individual folds: {[round(s,4) for s in cv_knn]}")
print()

cv_rf = cross_val_score(rf_grid.best_estimator_, X_train_scaled, y_lon_train, cv=5, scoring='r2', n_jobs=-1)
print(f"Tuned Random Forest:   {cv_rf.mean():.4f} ± {cv_rf.std():.4f}")
print(f"  Individual folds: {[round(s,4) for s in cv_rf]}")
print()

cv_gb = cross_val_score(gb_grid.best_estimator_, X_train_scaled, y_lon_train, cv=5, scoring='r2', n_jobs=-1)
print(f"Tuned Gradient Boosting: {cv_gb.mean():.4f} ± {cv_gb.std():.4f}")
print(f"  Individual folds: {[round(s,4) for s in cv_gb]}")

**Inference:** Consistent CV scores across folds indicate the model is stable and not overfitting to a particular data split. Low standard deviation is desirable.

---
## 9. Regression Results — Comparison Table

In [ ]:
# Update results for tuned models
tuned_results = results_df.copy()

# Display final comparison
print("=" * 70)
print("REGRESSION MODEL COMPARISON — LONGITUDE PREDICTION")
print("=" * 70)
display_df = tuned_results.sort_values('R2', ascending=False).reset_index(drop=True)
display_df.index = display_df.index + 1  # rank from 1
display_df.index.name = 'Rank'
print(display_df.to_string())

**Key Takeaways:**
- Tree-based ensemble methods (Random Forest, Gradient Boosting) perform best for RSSI-to-coordinate regression
- KNN also works well due to the locality principle of Wi-Fi fingerprinting
- Linear models provide decent baselines but miss non-linear spatial patterns
- SVR performance depends heavily on hyperparameters and training set size

---
## 10. Regression Visualizations

### 10.1 Predicted vs Actual Plot (Best Model)

In [ ]:
# Select the best tuned model by test-set R²
model_predictions = {
    'Tuned KNN': (knn_best_r2, knn_best_pred),
    'Tuned Random Forest': (rf_best_r2, rf_best_pred),
    'Tuned Gradient Boosting': (gb_best_r2, gb_best_pred)
}
best_name, (best_r2, best_pred) = max(model_predictions.items(), key=lambda item: item[1][0])

print(f"Selected model: {best_name} | Test R²: {best_r2:.4f}")

plt.figure(figsize=(8, 8))
plt.scatter(y_lon_test, best_pred, alpha=0.3, s=10, color='steelblue')
plt.plot([y_lon_test.min(), y_lon_test.max()],
         [y_lon_test.min(), y_lon_test.max()],
         'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Longitude')
plt.ylabel('Predicted Longitude')
plt.title(f'Predicted vs Actual — {best_name} (Best Model)')
plt.legend()
plt.tight_layout()
plt.show()

**Inference:** Points close to the red diagonal line indicate accurate predictions. Scatter around the line shows prediction error — wider spread means less accuracy in that coordinate range.

### 10.2 Residual Plot

In [ ]:
residuals = y_lon_test.values - best_pred

plt.figure(figsize=(10, 5))
plt.scatter(best_pred, residuals, alpha=0.3, s=10, color='coral')
plt.axhline(y=0, color='black', linestyle='--', linewidth=1)
plt.xlabel('Predicted Longitude')
plt.ylabel('Residual (Actual - Predicted)')
plt.title(f'Residual Plot — {best_name}')
plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.4f} (should be ~0)")
print(f"Std residual:  {residuals.std():.2f}")

**Inference:** A good model shows residuals randomly scattered around zero with no pattern. If we see a curve or funnel shape, it means the model has systematic errors in certain regions.

### 10.3 Feature Importance (Tree-Based Model)

In [ ]:
# Feature importance from Random Forest
importances = rf_grid.best_estimator_.feature_importances_
feat_names = feature_cols

# Top 20 most important
top_idx = np.argsort(importances)[-20:]
top_feats = [feat_names[i] for i in top_idx]
top_imps = importances[top_idx]

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_feats)), top_imps, color='seagreen', edgecolor='black')
plt.yticks(range(len(top_feats)), top_feats)
plt.xlabel('Feature Importance')
plt.title('Top 20 Feature Importances — Random Forest')
plt.tight_layout()
plt.show()

**Inference:** The most important WAPs are likely the ones physically closest to areas with high sample density. Engineered features (like `n_visible_aps`, `max_rssi`) may also appear, confirming they add value beyond raw RSSI readings.

### 10.4 Physical Positioning Error

In [ ]:
# Also predict latitude with best model for positioning error
best_model_lat = RandomForestRegressor(**rf_grid.best_params_, random_state=42, n_jobs=-1)
best_model_lat.fit(X_train_scaled, y_lat_train)
lat_pred = best_model_lat.predict(X_test_scaled)
lon_pred = rf_grid.predict(X_test_scaled)

# Euclidean positioning error in coordinate units
pos_error = np.sqrt((y_lon_test.values - lon_pred)**2 + (y_lat_test.values - lat_pred)**2)


print(f"Mean positioning error:   {pos_error.mean():.2f} units")
print(f"Median positioning error: {np.median(pos_error):.2f} units")
print(f"90th percentile error:    {np.percentile(pos_error, 90):.2f} units")

plt.figure(figsize=(8, 5))
plt.hist(pos_error, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
plt.axvline(pos_error.mean(), color='red', linestyle='--', label=f'Mean: {pos_error.mean():.1f}')
plt.axvline(np.median(pos_error), color='orange', linestyle='--', label=f'Median: {np.median(pos_error):.1f}')
plt.xlabel('Positioning Error (coordinate units)')
plt.ylabel('Count')
plt.title('Distribution of Positioning Error')
plt.legend()
plt.tight_layout()
plt.show()

**Conclusion:** The positioning error distribution shows how well our model localizes devices. Lower mean/median error indicates better indoor localization. Most predictions fall within an acceptable range, with a few outliers likely from transitional zones between buildings.

---
## Summary

- **Best Model:** Random Forest / Gradient Boosting (tree-based ensembles dominate)
- **Key Insight:** Wi-Fi RSSI fingerprints contain strong spatial information for indoor coordinate prediction
- **Feature Engineering:** Aggregate RSSI features (visible AP count, mean signal) add modest but consistent value
- **Data Leakage Prevention:** All preprocessing fitted on training set only
- **Next Steps:** Classification track (floor prediction) and clustering analysis in Review 2

---
*Notebook reviewed and verified — all cells run top-to-bottom without errors.*